Курс "Обработка текстов на естественном языке" 2026 (ЕКЛ) Тема 8b. Извлечение именованных сущностей. Библиотека Natasha
Natasha. Примеры извлечения именованных сущностей: даты, денежные единицы, адреса 

Документация: https://nbviewer.org/github/natasha/natasha/blob/master/docs.ipynb

In [88]:
from natasha import (
    Segmenter,
    MorphVocab,
    NewsEmbedding,
    NewsMorphTagger,
    NewsSyntaxParser,
    NewsNERTagger,  # Вот так: NER заглавными
    PER,
    LOC,
    ORG,
    Doc,
    NamesExtractor,
    DatesExtractor,
    AddrExtractor
)

In [89]:
segmenter = Segmenter()
morph_vocab = MorphVocab()
emb = NewsEmbedding()
ner_tagger = NewsNERTagger(emb)

In [90]:
# Экстракторы для конкретных типов
names_extractor = NamesExtractor(morph_vocab)
dates_extractor = DatesExtractor(morph_vocab)
addrs_extractor = AddrExtractor(morph_vocab)

In [91]:
text="""
Вчера, 12-го августа 2024 года, господин А. В. Петров выехал из Москвы в сторону Нижнего Новгорода.
В районе деревни Гадюкино, что под Тулой, он встретил инженера Иванова.
Они договорились провести совещание в Сбербанке в ближайший четверг, 15 августа.
В протоколе встречи указано, что ПАО "Газпром" и администрация города Санкт-Петербурга подпишут контракт уже в сентябре в Великом Устюге.
При этом лейтенант Сидорчук настаивал, что 5 мая 1945 года — ключевая дата, которую нельзя забывать в стенах МГУ.
"""

In [92]:
doc = Doc(text)

In [93]:
doc.segment(segmenter)
doc.tag_ner(ner_tagger)

In [94]:
# 3. Приведение имен к начальной форме (нормализация)
for span in doc.spans:
    span.normalize(morph_vocab)

print("--- Сущности (NER) ---")
for span in doc.spans:
    print(f"{span.text} -> {span.normal} ({span.type})")

--- Сущности (NER) ---
А. В. Петров -> А. В. Петров (PER)
Москвы -> Москвы (LOC)
Нижнего Новгорода -> Нижнего Новгорода (LOC)
Гадюкино -> Гадюкино (LOC)
Тулой -> Тулой (LOC)
Иванова -> Иванова (PER)
Сбербанке -> Сбербанке (ORG)
ПАО "Газпром" -> ПАО "Газпром" (ORG)
Санкт-Петербурга -> Санкт-Петербурга (LOC)
Великом Устюге -> Великом Устюге (LOC)
Сидорчук -> Сидорчук (PER)
МГУ -> МГУ (ORG)


In [95]:
print("\n--- Извлеченные даты ---")
dates = list(dates_extractor(text))
for date in dates:
    print(f"День: {date.fact.day}, Месяц: {date.fact.month}, Год: {date.fact.year}")


--- Извлеченные даты ---
День: None, Месяц: 8, Год: 2024
День: 15, Месяц: 8, Год: None
День: 5, Месяц: 5, Год: 1945


Не распознал 12-го августа 2024

In [96]:
# 5. Отдельный поиск адресов/городов
print("\n--- Населенные пункты ---")
addrs = list(addrs_extractor(text))
for addr in addrs:
    print(addr.fact)


--- Населенные пункты ---
AddrPart(value='Москвы', type=None)
AddrPart(value='Нижнего Новгорода', type=None)
AddrPart(value='Гадюкино', type='деревня')
AddrPart(value='Тулой', type=None)
AddrPart(value='Иванова', type=None)
AddrPart(value='Санкт-Петербурга', type='город')
